In [ ]:
import pandas as pd 
import numpy as np

In [ ]:
from sklearn.model_selection import train_test_split, KFold
from sklearn.feature_extraction import DictVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score

In [ ]:
data = "https://raw.githubusercontent.com/DataTalksClub/machine-learning-zoomcamp/main/cohorts/2026/data/course_lead_scoring_2026.csv"

In [ ]:
df = pd.read_csv(data)
df

In [ ]:
df.dtypes

In [ ]:
df.isnull().sum()

In [ ]:
for col in df.columns:
    if df[col].dtype in ['string']:
        df[col] = df[col].fillna('NA')
    elif df[col].dtype in ['int64', 'float64']:
        df[col] = df[col].fillna(0)

In [ ]:
df.isnull().sum()

In [ ]:
df_full_train, df_test = train_test_split(df, test_size=0.2, random_state=1)
df_train, df_val = train_test_split(df_full_train, test_size=0.25, random_state=1)

df_train = df_train.reset_index(drop=True)
df_val = df_val.reset_index(drop=True)
df_test = df_test.reset_index(drop=True)

y_train = df_train.converted.values
y_val = df_val.converted.values
y_test = df_test.converted.values

del df_train['converted']
del df_val['converted']
del df_test['converted']

In [ ]:
categorical = ['lead_source','industry','employment_status','location']
numerical = ['annual_income','number_of_courses_viewed','interaction_count','lead_score']

Which numerical variable (among the following 4) has the highest AUC?

- lead_score
- number_of_courses_viewed
- interaction_count
- annual_income

In [ ]:
for col in numerical:
  score = roc_auc_score(y_train, df_train[col])
  if score < 0.5:
    score = roc_auc_score(y_train, -df_train[col])
  print(f'{col}: {score:.3f}')

Apply one-hot-encoding using DictVectorizer and train the logistic regression with these parameters:

In [ ]:
X_train_dict = df_train[categorical + numerical].to_dict(orient="records")
X_val_dict = df_val[categorical + numerical].to_dict(orient="records")

dv = DictVectorizer(sparse=False)
X_train = dv.fit_transform(X_train_dict)
X_val = dv.transform(X_val_dict)

model = LogisticRegression(solver="liblinear", C=1.0, max_iter=1000)
model.fit(X_train, y_train)

y_pred = model.predict_proba(X_val)[:, 1]
auc_score = roc_auc_score(y_val, y_pred)

print(f"Model AUC: {auc_score:.3f}")


Choose the threshold with the smallest absolute difference between precision and recall.

- 0.43
- 0.63
- 0.73
- 0.83

In [ ]:
thresholds = np.linspace(0, 1, 101)
scores = []

for t in thresholds:
    predict_positive = (y_pred >= t)
    predict_negative = (y_pred < t)
    
    tp = (predict_positive & (y_val == 1)).sum()
    fp = (predict_positive & (y_val == 0)).sum()
    fn = (predict_negative & (y_val == 1)).sum()
    
    precision = tp / (tp + fp) if (tp + fp) > 0 else 0
    recall = tp / (tp + fn) if (tp + fn) > 0 else 0
    
    scores.append((t, precision, recall))

df_scores = pd.DataFrame(scores, columns=['threshold', 'precision', 'recall'])
df_scores['diff'] = abs(df_scores['precision'] - df_scores['recall'])
print(df_scores.sort_values(by='diff').head(10))

At which threshold F1 is maximal?

- 0.21
- 0.41
- 0.61
- 0.81

In [ ]:
f1_scores = []
for t, p, r in zip(df_scores["threshold"], df_scores["precision"], df_scores["recall"]):
    f1 = ((2 * p * r) / (p + r) if (p + r) > 0 else 0)
    f1_scores.append((t, f1))
df_f1 = pd.DataFrame(f1_scores, columns=["threshold", "f1_score"])
print(df_f1.sort_values(by="f1_score", ascending=False).head(1))

How large is standard deviation of the scores across different folds?

- 0.001
- 0.007
- 0.013
- 0.060

In [ ]:
kf = KFold(n_splits=5, shuffle=True, random_state=1)
auc_scores = []

for train_idx, val_idx in kf.split(df_full_train):
    df_train_cv = df_full_train.iloc[train_idx]
    df_val_cv = df_full_train.iloc[val_idx]

    y_train_cv = df_train_cv['converted'].values
    y_val_cv = df_val_cv['converted'].values

    X_train_dict = df_train_cv[categorical + numerical].to_dict(orient='records')
    X_val_dict = df_val_cv[categorical + numerical].to_dict(orient='records')

    dv = DictVectorizer(sparse=False)
    X_train_cv = dv.fit_transform(X_train_dict)
    X_val_cv = dv.transform(X_val_dict)

    model = LogisticRegression(solver='liblinear', C=1.0, max_iter=1000)
    model.fit(X_train_cv, y_train_cv)

    y_pred_cv = model.predict_proba(X_val_cv)[:, 1]
    auc = roc_auc_score(y_val_cv, y_pred_cv)
    auc_scores.append(auc)

print(f"Mean AUC: {np.mean(auc_scores):.3f} , Standard Deviation: {np.std(auc_scores):.3f}")

Which C leads to the best mean ROC AUC?

- 0.000001
- 0.001
- 1

In [ ]:
C_values = [0.000001, 0.001, 0.1, 1, 10]

for C in C_values:
    auc_scores = []
    for train_idx, val_idx in kf.split(df_full_train):
        df_train_cv = df_full_train.iloc[train_idx]
        df_val_cv = df_full_train.iloc[val_idx]
        
        y_train_cv = df_train_cv['converted'].values
        y_val_cv = df_val_cv['converted'].values
        
        X_train_dict = df_train_cv[categorical + numerical].to_dict(orient='records')
        X_val_dict = df_val_cv[categorical + numerical].to_dict(orient='records')
        
        dv = DictVectorizer(sparse=False)
        X_train_cv = dv.fit_transform(X_train_dict)
        X_val_cv = dv.transform(X_val_dict)
        
        model = LogisticRegression(solver='liblinear', C=C, max_iter=1000)
        model.fit(X_train_cv, y_train_cv)
        
        y_pred_cv = model.predict_proba(X_val_cv)[:, 1]
        auc = roc_auc_score(y_val_cv, y_pred_cv)
        auc_scores.append(auc)
        
    print(f"C: {C} , Mean AUC: {np.mean(auc_scores):.3f} ,Std: {np.std(auc_scores):.3f}")